# MLOps Training 2026/2027 — Task 2
## Notebook 6 — Train, Tune, Evaluate

### Objective
1. Load the preprocessed feature matrices from Notebook 5.
2. Establish a **baseline** — the floor any real model must beat.
3. Train and tune `HistGradientBoostingClassifier` using **only the validation split**.
4. Touch the **test split once**, at the very end.
- `artifacts/notebook_05/feature_config.json`, `feature_names.json`, `feature_list.csv`

### Why HistGradientBoosting?
- Handles NaN natively (no imputation overhead).
- `class_weight='balanced'` for the imbalanced problem.
- `l2_regularization` and `min_samples_leaf` give explicit overfitting control.
- Faster than RandomForest on large data.

### Input Artifacts
- `artifacts/notebook_05/X_train.npy`, `X_validation.npy`, `X_test.npy`
- `artifacts/notebook_05/y_train.npy`, `y_validation.npy`, `y_test.npy`
- `artifacts/notebook_05/feature_names.json`, `feature_config.json`

### Output Artifacts
- `artifacts/notebook_06/baseline_model.joblib`
- `artifacts/notebook_06/final_model.joblib`
- `artifacts/notebook_06/tuning_results.csv`
- `artifacts/notebook_06/results_summary.json`
- `artifacts/notebook_06/serving_config.json`

## 1. Imports & Configuration

In [1]:
from pathlib import Path
import json
import itertools
import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve,
)

RANDOM_STATE   = 42
PRIMARY_METRIC = "average_precision"  # threshold-independent PR-AUC

INPUT_DIR  = Path.cwd() / "artifacts" / "notebook_05"
OUTPUT_DIR = Path.cwd() / "artifacts" / "notebook_06"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_DIR.exists(), f"Notebook 5 artifacts not found: {INPUT_DIR.resolve()}"
print("Input  dir:", INPUT_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())

Input  dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_05
Output dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_06


## 2. Load Preprocessed Artifacts

Test labels are loaded now so shapes can be validated — but **`y_test` is not looked at** until Section 10.

In [2]:
X_train      = np.load(INPUT_DIR / "X_train.npy")
X_validation = np.load(INPUT_DIR / "X_validation.npy")
X_test       = np.load(INPUT_DIR / "X_test.npy")

y_train      = np.load(INPUT_DIR / "y_train.npy")
y_validation = np.load(INPUT_DIR / "y_validation.npy")
y_test       = np.load(INPUT_DIR / "y_test.npy")

with open(INPUT_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

with open(INPUT_DIR / "feature_config.json") as f:
    feature_config = json.load(f)

TARGET = feature_config["target"]

assert X_train.shape[0] == len(y_train)
assert X_validation.shape[0] == len(y_validation)
assert X_test.shape[0] == len(y_test)
assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1] == len(feature_names)

print(f"Target   : {TARGET}")
print(f"Features : {len(feature_names)}")
print(f"Train    : X={X_train.shape}, y={y_train.shape}")
print(f"Val      : X={X_validation.shape}, y={y_validation.shape}")
print(f"Test     : X={X_test.shape}, y={y_test.shape}")

Target   : is_late
Features : 53
Train    : X=(67533, 53), y=(67533,)
Val      : X=(14471, 53), y=(14471,)
Test     : X=(14472, 53), y=(14472,)


## 3. Confirm Class Imbalance

In [3]:
train_pos_rate = float(y_train.mean())
val_pos_rate   = float(y_validation.mean())

for name, y in [("train", y_train), ("validation", y_validation)]:
    print(f"{name:<12} n={len(y):>7,}  positive ({TARGET}=1) rate: {y.mean():.2%}")

imbalance_ratio = (1 - train_pos_rate) / train_pos_rate
print(f"\nNeg:pos ratio ≈ {imbalance_ratio:.1f} : 1")
print("→ Accuracy alone is a misleading metric. Use PR-AUC / ROC-AUC / F1.")

train        n= 67,533  positive (is_late=1) rate: 9.02%
validation   n= 14,471  positive (is_late=1) rate: 5.33%

Neg:pos ratio ≈ 10.1 : 1
→ Accuracy alone is a misleading metric. Use PR-AUC / ROC-AUC / F1.


## 4. Evaluation Helper

**Primary metric**: Average Precision (PR-AUC) — threshold-independent, rewards models that
rank positives (late orders) above negatives.  
**Secondary**: ROC-AUC, F1, precision, recall — reported for context.

In [4]:
def evaluate(y_true, y_proba, threshold=0.5, label=""):
    y_pred = (y_proba >= threshold).astype(int)
    metrics = {
        "average_precision": float(average_precision_score(y_true, y_proba)),
        "roc_auc"          : float(roc_auc_score(y_true, y_proba)),
        "f1"               : float(f1_score(y_true, y_pred, zero_division=0)),
        "precision"        : float(precision_score(y_true, y_pred, zero_division=0)),
        "recall"           : float(recall_score(y_true, y_pred, zero_division=0)),
        "accuracy"         : float(accuracy_score(y_true, y_pred)),
    }
    if label:
        print(f"\n  {label}")
        for k, v in metrics.items():
            print(f"    {k:<22}: {v:.4f}")
    return metrics

print(f"Primary metric: {PRIMARY_METRIC}")

Primary metric: average_precision


## 5. Baseline Model

A `DummyClassifier(strategy='most_frequent')` always predicts the majority class
(on-time). This is the **floor** — any real model must beat it.

In [5]:
baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

baseline_val_proba   = baseline.predict_proba(X_validation)[:, 1]
baseline_val_metrics = evaluate(y_validation, baseline_val_proba,
                                label="Baseline (DummyClassifier) — validation")


  Baseline (DummyClassifier) — validation
    average_precision     : 0.0533
    roc_auc               : 0.5000
    f1                    : 0.0000
    precision             : 0.0000
    recall                : 0.0000
    accuracy              : 0.9467


## 6. HistGradientBoosting Grid Search

Each candidate is fit on `X_train`, scored on `X_validation`.  
The test split is not touched until Section 10.

**Regularisation choices:**
- `min_samples_leaf` ≥ 15 — prevents leaves from fitting just a handful of late orders.
- `l2_regularization` ∈ {1.0, 2.0} — explicit L2 penalty on leaf values.
- `class_weight='balanced'` — compensates for 10:1 class imbalance automatically.

In [6]:
hgb_grid = {
    "learning_rate"    : [0.05, 0.08, 0.1],
    "max_iter"         : [400, 600],
    "max_leaf_nodes"   : [31, 47],
    "min_samples_leaf" : [15, 20, 30],
    "l2_regularization": [1.0, 2.0],
}

hgb_combos = [
    dict(zip(hgb_grid.keys(), vals))
    for vals in itertools.product(*hgb_grid.values())
]
print(f"Evaluating {len(hgb_combos)} combinations on the validation split ...\n")

tuning_rows = []
for params in hgb_combos:
    m = HistGradientBoostingClassifier(
        **params, class_weight="balanced", random_state=RANDOM_STATE
    )
    m.fit(X_train, y_train)
    val_proba = m.predict_proba(X_validation)[:, 1]
    val_m     = evaluate(y_validation, val_proba)
    tuning_rows.append({
        "params": str(params),
        **{k: val_m[k] for k in val_m},
        "_model": m,
    })
    print(f"  {params}  →  {PRIMARY_METRIC}={val_m[PRIMARY_METRIC]:.4f}  roc_auc={val_m['roc_auc']:.4f}")

tuning_rows.sort(key=lambda r: r[PRIMARY_METRIC], reverse=True)

print(f"\nTop 5:")
for row in tuning_rows[:5]:
    print(f"  {PRIMARY_METRIC}={row[PRIMARY_METRIC]:.4f}  {row['params']}")

Evaluating 72 combinations on the validation split ...

  {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 15, 'l2_regularization': 1.0}  →  average_precision=0.1070  roc_auc=0.6459
  {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 15, 'l2_regularization': 2.0}  →  average_precision=0.1059  roc_auc=0.6462
  {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 1.0}  →  average_precision=0.1074  roc_auc=0.6437
  {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 2.0}  →  average_precision=0.1060  roc_auc=0.6446
  {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'l2_regularization': 1.0}  →  average_precision=0.1041  roc_auc=0.6427
  {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'l2_regularization': 2.0}  →  average_precision=0.1073 

## 7. Select Best Model & Compare with Baseline

In [7]:
best_row          = tuning_rows[0]
final_model       = best_row["_model"]
final_val_metrics = {k: best_row[k] for k in ("average_precision","roc_auc","f1","precision","recall","accuracy")}
best_params_str   = best_row["params"]

print(f"Best params: {best_params_str}")

header = f"{'metric':<22}{'baseline':>12}{'tuned':>12}{'delta':>12}"
print(f"\n{header}")
for k in baseline_val_metrics:
    b = baseline_val_metrics[k]
    t = final_val_metrics[k]
    print(f"{k:<22}{b:>12.4f}{t:>12.4f}{t-b:>+12.4f}")

assert final_val_metrics[PRIMARY_METRIC] > baseline_val_metrics[PRIMARY_METRIC], \
    "Tuned model does not beat the baseline — investigate before touching test set!"
print(f"\n✓ Tuned model beats the baseline on {PRIMARY_METRIC}.")

Best params: {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 47, 'min_samples_leaf': 15, 'l2_regularization': 2.0}

metric                    baseline       tuned       delta
average_precision           0.0533      0.1103     +0.0570
roc_auc                     0.5000      0.6475     +0.1475
f1                          0.0000      0.1734     +0.1734
precision                   0.0000      0.1165     +0.1165
recall                      0.0000      0.3385     +0.3385
accuracy                    0.9467      0.8280     -0.1187

✓ Tuned model beats the baseline on average_precision.


## 8. Threshold Optimisation (F1-max on Validation)

In [8]:
val_proba = final_model.predict_proba(X_validation)[:, 1]
prec_arr, rec_arr, thresholds = precision_recall_curve(y_validation, val_proba)
f1_arr    = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
best_idx  = int(np.argmax(f1_arr))
optimal_threshold = float(thresholds[best_idx])

print(f"Optimal threshold (F1-max on validation): {optimal_threshold:.4f}")
print(f"Val F1 at optimal threshold            : {f1_arr[best_idx]:.4f}")

Optimal threshold (F1-max on validation): 0.6572
Val F1 at optimal threshold            : 0.1793


## 9. Feature Importance (Permutation, on Validation Set)

In [9]:
print("Computing permutation importance on validation (n_repeats=5) ...")
pi = permutation_importance(
    final_model, X_validation, y_validation,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1,
)
pi_mean = pi.importances_mean
top_idx = np.argsort(pi_mean)[::-1][:15]

print(f"\nTop 15 features (permutation importance — val AP drop when shuffled):")
top_features = []
for rank, i in enumerate(top_idx, 1):
    print(f"  {rank:>2}. {feature_names[i]:<50} {pi_mean[i]:.4f}")
    top_features.append({"feature": feature_names[i], "importance": float(pi_mean[i])})

Computing permutation importance on validation (n_repeats=5) ...

Top 15 features (permutation importance — val AP drop when shuffled):
   1. numeric__customer_zip_code_prefix                  0.0384
   2. numeric__estimated_delivery_days                   0.0219
   3. numeric__distance_km                               0.0110
   4. numeric__total_freight_value                       0.0074
   5. numeric__seller_zip_code_prefix                    0.0056
   6. numeric__approval_delay_hours                      0.0026
   7. numeric__item_count                                0.0023
   8. numeric__seller_city                               0.0013
   9. categorical__customer_state_PE                     0.0013
  10. numeric__customer_city                             0.0010
  11. numeric__purchase_hour                             0.0010
  12. numeric__unique_sellers                            0.0008
  13. categorical__customer_state_SP                     0.0006
  14. numeric__purchase_weekday 

## 10. Final Test Set Evaluation — Touched Once

> ⚠️ This is the **only cell** in the notebook that reads `y_test` or scores against `X_test`.

In [10]:
test_proba = final_model.predict_proba(X_test)[:, 1]

final_test_default = evaluate(y_test, test_proba, threshold=0.5,
                              label="Test @ 0.50 threshold")
final_test_optimal = evaluate(y_test, test_proba, threshold=optimal_threshold,
                              label=f"Test @ {optimal_threshold:.4f} threshold (F1-optimal on val)")

cm_default = confusion_matrix(y_test, (test_proba >= 0.5).astype(int))
cm_optimal = confusion_matrix(y_test, (test_proba >= optimal_threshold).astype(int))

print(f"\nConfusion matrix @ 0.50 threshold (rows=actual, cols=predicted):")
print(f"  {cm_default}")
print(f"\nConfusion matrix @ {optimal_threshold:.4f} threshold:")
print(f"  {cm_optimal}")

gap = final_val_metrics[PRIMARY_METRIC] - final_test_default[PRIMARY_METRIC]
print(f"\nValidation → Test gap ({PRIMARY_METRIC}): {gap:.4f}  "
      f"({'✓ acceptable' if gap < 0.04 else '⚠ large'})")


  Test @ 0.50 threshold
    average_precision     : 0.1072
    roc_auc               : 0.6265
    f1                    : 0.1674
    precision             : 0.1136
    recall                : 0.3180
    accuracy              : 0.7910

  Test @ 0.6572 threshold (F1-optimal on val)
    average_precision     : 0.1072
    roc_auc               : 0.6265
    f1                    : 0.1717
    precision             : 0.1351
    recall                : 0.2354
    accuracy              : 0.8500

Confusion matrix @ 0.50 threshold (rows=actual, cols=predicted):
  [[11143  2373]
 [  652   304]]

Confusion matrix @ 0.6572 threshold:
  [[12076  1440]
 [  731   225]]

Validation → Test gap (average_precision): 0.0031  (✓ acceptable)


## 11. Save Tuning Results CSV

In [11]:
tuning_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "_model"}
    for row in tuning_rows
]).sort_values(PRIMARY_METRIC, ascending=False)

tuning_df.to_csv(OUTPUT_DIR / "tuning_results.csv", index=False)
print(f"✓ Saved tuning_results.csv ({len(tuning_df)} candidates)")
tuning_df[["params", PRIMARY_METRIC, "roc_auc", "f1"]].head(10)

✓ Saved tuning_results.csv (72 candidates)


,params,average_precision,roc_auc,f1
0,"{'learning_rate': 0.05, 'max_iter': 400, 'max_...",0.110309,0.647549,0.173364
1,"{'learning_rate': 0.05, 'max_iter': 600, 'max_...",0.110309,0.647549,0.173364
2,"{'learning_rate': 0.08, 'max_iter': 400, 'max_...",0.109641,0.647722,0.170079
3,"{'learning_rate': 0.08, 'max_iter': 600, 'max_...",0.109641,0.647722,0.170079
4,"{'learning_rate': 0.1, 'max_iter': 400, 'max_l...",0.109462,0.648650,0.168344
5,"{'learning_rate': 0.1, 'max_iter': 600, 'max_l...",0.109462,0.648650,0.168344
6,"{'learning_rate': 0.1, 'max_iter': 400, 'max_l...",0.109235,0.648475,0.171927
7,"{'learning_rate': 0.1, 'max_iter': 600, 'max_l...",0.109235,0.648475,0.171927
8,"{'learning_rate': 0.08, 'max_iter': 400, 'max_...",0.109150,0.648333,0.171940
9,"{'learning_rate': 0.08, 'max_iter': 600, 'max_...",0.109150,0.648333,0.171940


## 12. Persist Models & Results Summary

In [12]:
# Save models
joblib.dump(baseline,    OUTPUT_DIR / "baseline_model.joblib")
joblib.dump(final_model, OUTPUT_DIR / "final_model.joblib")
print("✓ baseline_model.joblib")
print("✓ final_model.joblib")

results_summary = {
    "target"                                  : TARGET,
    "primary_metric"                          : PRIMARY_METRIC,
    "train_positive_rate"                     : float(train_pos_rate),
    "baseline_model"                          : "DummyClassifier(strategy='most_frequent')",
    "families_searched"                       : ["HistGradientBoosting"],
    "n_candidates_evaluated"                  : len(tuning_rows),
    "final_model_family"                      : "HistGradientBoosting",
    "final_model_params"                      : best_params_str,
    "final_model_uses_dense_input"            : True,
    "optimal_decision_threshold"              : optimal_threshold,
    "baseline_validation_metrics"             : baseline_val_metrics,
    "final_validation_metrics"                : final_val_metrics,
    "final_test_metrics_default_threshold"    : final_test_default,
    "final_test_metrics_tuned_threshold"      : final_test_optimal,
    "test_confusion_matrix_default_threshold" : cm_default.tolist(),
    "test_confusion_matrix_tuned_threshold"   : cm_optimal.tolist(),
    "top_features"                            : top_features,
    "n_features"                              : len(feature_names),
    "validation_test_gap_primary_metric"      : round(gap, 6),
    "known_limitations": [
        "Temporal drift: val/test orders are later in time than train — some gap is irreducible.",
        "A broader Bayesian / random search would likely improve absolute AP further.",
        "class_weight='balanced' shifts predicted scores; late_probability is a rank, not a calibrated chance of being late.",
        "Accuracy is dominated by the on-time class and is not the serving quality metric.",
    ],
    "random_state": RANDOM_STATE,
}

with open(OUTPUT_DIR / "results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print("✓ results_summary.json")

serving_config = {
    "model_version": "notebook06-histgbm-checkout-v2",
    "prediction_point": feature_config.get("prediction_point"),
    "primary_output": "late_probability",
    "optional_output": "predicted_late",
    "decision_threshold": optimal_threshold,
    "threshold_objective": "f1_max_on_validation",
    "threshold_is_policy": True,
    "score_is_calibrated_probability": False,
    "note": (
        "Return late_probability as the main API output. predicted_late is "
        "probability >= decision_threshold and can change without retraining. "
        "Do not advertise accuracy. Reject request fields listed in "
        "feature_config.api_forbidden_fields (delivery dates, reviews, "
        "order_approved_at, approval_delay_hours)."
    ),
    "artifacts": {
        "model": str(OUTPUT_DIR / "final_model.joblib"),
        "baseline": str(OUTPUT_DIR / "baseline_model.joblib"),
        "preprocessor": str(INPUT_DIR / "preprocessor.joblib"),
        "target_encoder": str(INPUT_DIR / "target_encoder.joblib"),
        "feature_names": str(INPUT_DIR / "feature_names.json"),
        "feature_config": str(INPUT_DIR / "feature_config.json"),
    },
    "n_features": len(feature_names),
    "random_state": RANDOM_STATE,
}

with open(OUTPUT_DIR / "serving_config.json", "w") as f:
    json.dump(serving_config, f, indent=2)
print("✓ serving_config.json")

✓ baseline_model.joblib
✓ final_model.joblib
✓ results_summary.json


## 13. Verification

In [13]:
loaded_model  = joblib.load(OUTPUT_DIR / "final_model.joblib")
check_proba   = loaded_model.predict_proba(X_validation)[:, 1]
assert np.allclose(check_proba, val_proba, atol=1e-6), "Reload mismatch!"
print("✓ Reloaded model produces identical probabilities.")

✓ Reloaded model produces identical probabilities.


## Summary

| Step | What happened |
|---|---|
| Baseline | DummyClassifier(most_frequent): PR-AUC ≈ 0.053, always predicts on-time |
| Grid search | 72 HistGBM configurations evaluated on validation only |
| Best model | HistGradientBoosting with class_weight='balanced', l2 regularisation |
| Threshold | F1-optimal threshold tuned on validation, applied once to test |
| Test set | Touched exactly once at the end |

### Main artifacts
- `artifacts/notebook_06/baseline_model.joblib`
- `artifacts/notebook_06/final_model.joblib`
- `artifacts/notebook_06/tuning_results.csv`
- `artifacts/notebook_06/results_summary.json`
- `artifacts/notebook_06/serving_config.json` — threshold, prediction point, and which artifacts a future API must load (never refit)